# NB07 - Deployment Dashboard Analisis Sentimen ACSC EWA

Notebook ini menjalankan dashboard Streamlit (`app_final_revisi.py`) di Google Colab.

Perbaikan pada versi ini dibandingkan versi sebelumnya:
1. **Konsistensi praproses pelatihan-inferensi**: teks masukan pada panel Analisis
   Ulasan Baru melewati pipeline praproses yang sama dengan pelatihan (case folding,
   penghapusan noise, normalisasi karakter berulang tiga-atau-lebih menjadi dua,
   normalisasi kata non-baku dengan kamus dan protected words) sebelum tokenisasi.
2. **Deteksi relevansi aspek**: sentimen hanya diprediksi pada kategori aspek yang
   terdeteksi relevan (heuristik leksikon), karena model ACSC mengasumsikan aspek
   sudah diketahui. Mode prediksi seluruh aspek tetap tersedia sebagai pilihan.
3. **Angka hasil akhir**: seluruh metrik pada dashboard mengikuti hasil final
   (IndoBERT Macro-F1 0,8762 +/- 0,0137 rata-rata lima seed; checkpoint terbaik
   0,8843; baseline SVM 0,8454; data uji 454 baris; dataset final 2.266 baris).
4. **Tampilan formal**: tanpa emoji dan ornamen; bahasa akademis.

Kebutuhan file di Google Drive (folder proyek):
- `app_final_revisi.py` (dashboard)
- `best_model_indobert.pt` (checkpoint terbaik, kriteria Val Weighted-F1)
- `kamuskatabaku.xlsx` (kamus normalisasi 4.975 entri)


## Sel 1 - Instalasi dependensi


In [ ]:
!pip -q install streamlit==1.35.0 transformers torch plotly openpyxl
!npm -q install -g localtunnel
print("Instalasi selesai.")


## Sel 2 - Hubungkan Google Drive dan salin berkas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DIR_PROYEK = "/content/drive/MyDrive/TA_ACSC_EWA"   # sesuaikan bila berbeda

for f in ["app_final_revisi.py", "kamuskatabaku.xlsx"]:
    src = os.path.join(DIR_PROYEK, f)
    if os.path.exists(src):
        shutil.copy(src, f"/content/{f}")
        print(f"Tersalin : {f}")
    else:
        print(f"TIDAK DITEMUKAN: {src} - salin manual bila perlu")

src_model = os.path.join(DIR_PROYEK, "05_modeling/indobert/best_model_indobert.pt")
if os.path.exists(src_model):
    shutil.copy(src_model, "/content/best_model_indobert.pt")
    print("Tersalin : best_model_indobert.pt")
else:
    print(f"TIDAK DITEMUKAN: {src_model} - sesuaikan path checkpoint Anda")


## Sel 3 - Verifikasi berkas sebelum menjalankan


In [ ]:
import os
wajib = ["app_final_revisi.py", "best_model_indobert.pt", "kamuskatabaku.xlsx"]
for f in wajib:
    status = "ADA" if os.path.exists(f"/content/{f}") else "TIDAK ADA"
    ukuran = f"{os.path.getsize(f'/content/{f}')/1e6:.1f} MB" if status == "ADA" else "-"
    print(f"{f:<28} {status:<10} {ukuran}")
print()
print("Catatan: dashboard tetap dapat berjalan tanpa kamus (memakai kamus bawaan")
print("ringkas), namun untuk kesetaraan penuh dengan pelatihan sediakan kamus asli.")


## Sel 4 - Jalankan dashboard

Setelah sel ini dijalankan:
1. Salin alamat IP yang tercetak (baris "Password/Endpoint IP").
2. Buka tautan localtunnel yang muncul.
3. Tempel alamat IP tersebut pada halaman yang meminta kata sandi terowongan.


In [ ]:
import subprocess, time

with open("/content/streamlit_log.txt", "w") as log:
    proc = subprocess.Popen(
        ["streamlit", "run", "/content/app_final_revisi.py",
         "--server.port", "8501", "--server.headless", "true"],
        stdout=log, stderr=log)
time.sleep(6)
print("Streamlit berjalan pada port 8501.")

!echo "Password/Endpoint IP:"
!curl -s https://loca.lt/mytunnelpassword ; echo
!npx localtunnel --port 8501


## Sel 5 - Menghentikan dashboard

Hentikan sel 4 (tombol stop), lalu jalankan sel ini untuk memastikan proses berhenti.


In [ ]:
!pkill -f streamlit || true
print("Proses Streamlit dihentikan.")
